In [35]:
from pathlib import Path
import pandas as pd

In [113]:
######################################
## Steel – Hypothèses ##
######################################

# Facteurs H2 : tH2 / tSteel
H2_PER_TSTEEL = {
    "EAF": 0.0,
    "DRI-EAF_H2": 0.075,
    "DRI-EAF_CH4": 0.0,
    "BF-BOF": 0.0,
}

# Ordre d’affichage des technos (modifiable)
TECHNO_ORDER = ["BF-BOF", "EAF", "DRI-EAF_CH4", "DRI-EAF_H2"]

# Horizon complet
START_YEAR = 2020
END_YEAR   = 2050  # inclus

# Sortie
STEEL_CSV  = Path("steel.csv")
STEEL_COLS = ["area","techno","year","production_yearly","hydrogen_demand","source"]


In [114]:
# CHUNK 2 — Règles (corrigées)
#   - constant => prod_end doit être None (ou identique à prod_start)
#   - linéaire => prod_end obligatoire
#---------------------------------------------
ABS_RULES = [
    ############## FRANCE ################
    {"area": "FR", "techno": "BF-BOF", "start_year": 2020, "end_year": 2030, "mode": "constant",
     "prod_start": 5.92e6, "prod_end": None, "source": "ADEME - Plan de Transition Sectoriel de l'Acier"},

    {"area": "FR", "techno": "BF-BOF", "start_year": 2030, "end_year": 2051, "mode": "constant",
     "prod_start": 3.42e6, "prod_end": None, "source": "France Stratégie - Rapports coûts abattements"},

    {"area": "FR", "techno": "EAF", "start_year": 2040, "end_year": 2051, "mode": "constant",
     "prod_start": 4.084e6, "prod_end": None, "source": "ADEME - Plan de Transition Sectoriel de l'Acier"},

    {"area": "FR", "techno": "DRI-EAF_CH4", "start_year": 2030, "end_year": 2040, "mode": "constant",
     "prod_start": 2.5e6, "prod_end": 2.5e6, "source": "France Stratégie - Rapports coûts abattements"},

    {"area": "FR", "techno": "DRI-EAF_H2", "start_year": 2030, "end_year": 2040, "mode": "constant",
     "prod_start": 5.0e6, "prod_end": None, "source": "France Stratégie - Rapports coûts abattements"},

    ############## ALLEMAGNE ################
    {"area": "DE", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 25.9e6, "prod_end": None, "source": "Stahl - Facts and Figures about the steel industry in Germany"},

    {"area": "DE", "techno": "EAF", "start_year": 2040, "end_year": 2051, "mode": "constant",
     "prod_start": 11.1e6, "prod_end": None, "source": "Stahl - Facts and Figures about the steel industry in Germany"},

    ############## BELGIUM ################

    {"area": "BE", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 5.25e6, "prod_end": None, "source": "Steel BEL - Rapports 2024"},

    {"area": "BE", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 1.75e6, "prod_end": None, "source": "Steel BEL - Rapports 2024"},

    ############## SPAIN ################

    {"area": "ES", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 3.166e6, "prod_end": None, "source": "Energy Transitions Commissions"},

    {"area": "ES", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 8.102e6, "prod_end": None, "source": "Energy Transitions Commissions"},

    ############## ITALY ################

    {"area": "IT", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 3.15, "prod_end": None, "source": "Brief - Cassa Depositi e Prestiti"},

    {"area": "IT", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 17.85e6, "prod_end": None, "source": "Brief - Cassa Depositi e Prestiti"},

    ############## SLOVAKIA ################

    {"area": "SK", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 4.1e6, "prod_end": None, "source": "JRC - 2023"},

    {"area": "SK", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 0.245e6, "prod_end": None, "source": "JRC - 2023"},

    ############## SLOVENIA ################

    {"area": "SL", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 0.550e6, "prod_end": None, "source": "JRC - 2023"},

    ############## SWEDEN ################

    {"area": "SE", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 2.950e6, "prod_end": None, "source": "JRC - 2023"},

    {"area": "SE", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 1.350e6, "prod_end": None, "source": "JRC - 2023"},

    ############## ROMANIA ################



    {"area": "RO", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 0.9095e6, "prod_end": None, "source": "JRC - 2023"},

    {"area": "RO", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 0.8e6, "prod_end": None, "source": "JRC - 2023"},

 ############## PORTUGAL ################

    {"area": "PT", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 2.0e6, "prod_end": None, "source": "JRC - 2023"},

     ############## POLAND ################

    {"area": "PL", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 3.123e6, "prod_end": None, "source": "JRC - 2023"},

    {"area": "PL", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 3.276e6, "prod_end": None, "source": "JRC - 2023"},

    ############## NETHERLANDS ################

    {"area": "NE", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 4.8553e6, "prod_end": None, "source": "JRC - 2023"},

    ############## MALTA ################

    ############## LATVIA ################

    ############## LUXEMBOURG ################

    {"area": "LU", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 1.899e6, "prod_end": None, "source": "JRC - 2023"},

############## LITHUANIA ################

############## IRELAND ################

    ############## HUNGARY ################

    {"area": "HU", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 0.2498e6, "prod_end": None, "source": "JRC - 2023"},

    {"area": "HU", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 0.2278e6, "prod_end": None, "source": "JRC - 2023"},

    ############## CROATIA ################

    {"area": "CR", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 0.2124e6, "prod_end": None, "source": "JRC - 2023"},

    ############## FINLAND ################

    {"area": "HU", "techno": "BF-BOF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 2.342e6, "prod_end": None, "source": "JRC - 2023"},

    {"area": "HU", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 1.435e6, "prod_end": None, "source": "JRC - 2023"},

    ############## GREECE ################

    {"area": "HU", "techno": "EAF", "start_year": 2020, "end_year": 2051, "mode": "constant",
     "prod_start": 1.134e6, "prod_end": None, "source": "JRC - 2023"},

]

In [115]:
# CHUNK 3 — Interpolation production (strict)
#---------------------------------------------
def interp_prod(mode: str, p0: float, p1: float | None, year: int, y0: int, y1: int) -> float:
    if mode == "constant":
        return float(p0)
    if mode == "linear":
        if p1 is None:
            raise ValueError("prod_end requis en mode linear")
        frac = (year - y0) / (y1 - y0)  # end_year exclusif
        return float(p0) + frac * (float(p1) - float(p0))
    raise ValueError(f"mode inconnu: {mode}")

In [116]:
# CHUNK 4 — Générer production active (uniquement sur plages)
#---------------------------------------------
def generate_active_production(rules: list[dict]) -> pd.DataFrame:
    rows = []
    for r in rules:
        # validations minimales
        if r["mode"] == "linear" and r.get("prod_end", None) is None:
            raise ValueError(f"prod_end manquant pour une règle linear: {r}")
        if r["mode"] == "constant" and r.get("prod_end", None) not in (None, r["prod_start"]):
            # on tolère prod_end==prod_start, sinon on force à corriger
            raise ValueError(f"En mode constant, prod_end doit être None (ou =prod_start). Règle: {r}")

        y0, y1 = int(r["start_year"]), int(r["end_year"])  # end exclusive
        for y in range(y0, y1):
            prod = interp_prod(r["mode"], r["prod_start"], r.get("prod_end", None), y, y0, y1)
            rows.append({
                "area": r["area"],
                "techno": r["techno"],
                "year": y,
                "production_yearly": prod,
                "source": r["source"]
            })
    return pd.DataFrame(rows)

In [117]:
# CHUNK 5 — Compléter 2020..2050 pour chaque (area, techno) présent
#   -> production=0 si absent
#   -> source="filled_zero" si absent
#---------------------------------------------
def complete_years(df: pd.DataFrame, start_year: int = START_YEAR, end_year: int = END_YEAR) -> pd.DataFrame:
    df = df.copy()
    years = list(range(start_year, end_year + 1))  # end inclusive

    # couples (area, techno) présents
    area_tech = df[["area", "techno"]].drop_duplicates()

    # grille complète (area, techno, year)
    full_index = (
        area_tech.assign(_k=1)
        .merge(pd.DataFrame({"year": years, "_k": 1}), on="_k")
        .drop(columns="_k")
    )

    df_full = full_index.merge(df, on=["area", "techno", "year"], how="left")

    df_full["production_yearly"] = pd.to_numeric(df_full["production_yearly"], errors="coerce").fillna(0.0)
    df_full["source"] = df_full["source"].fillna("None")

    return df_full

In [118]:
# CHUNK 6 — Calcul H2 DEMAND (vectorisé, strict)
#---------------------------------------------
def compute_hydrogen_demand(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["area"] = df["area"].astype("string")
    df["techno"] = df["techno"].astype("string")
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    df["production_yearly"] = pd.to_numeric(df["production_yearly"], errors="coerce").fillna(0.0).astype(float)

    df["h2_factor"] = df["techno"].map(H2_PER_TSTEEL)
    if df["h2_factor"].isna().any():
        missing = df.loc[df["h2_factor"].isna(), "techno"].unique()
        raise ValueError(f"Facteur H2 manquant pour techno(s): {missing}")

    df["hydrogen_demand"] = df["production_yearly"] * df["h2_factor"]
    return df.drop(columns=["h2_factor"])

In [119]:
# CHUNK 7 — Tri d’affichage: area -> techno (ordre custom) -> year
#---------------------------------------------
def sort_for_display(df: pd.DataFrame, techno_order: list[str] | None = None) -> pd.DataFrame:
    df = df.copy()
    df["area"] = df["area"].astype("string")
    df["techno"] = df["techno"].astype("string")
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

    if techno_order is None:
        df["techno_sort"] = df["techno"]
        df = df.sort_values(["area", "techno_sort", "year"]).drop(columns=["techno_sort"])
    else:
        order_map = {t: i for i, t in enumerate(techno_order)}
        df["techno_sort"] = df["techno"].map(order_map).fillna(10_000).astype(int)
        df["techno_fallback"] = df["techno"]
        df = df.sort_values(["area", "techno_sort", "techno_fallback", "year"]).drop(
            columns=["techno_sort", "techno_fallback"]
        )

    return df.reset_index(drop=True)

In [120]:
# CHUNK 8 — Pipeline complet: build + write steel.csv
#   - génère production active
#   - complète années 2020..2050 pour chaque (area, techno)
#   - calcule H2 demand
#   - trie: pays -> techno -> années
#   - écrit steel.csv (écrase le fichier : output déterministe)
#---------------------------------------------
def build_and_write_steel(rules: list[dict], techno_order: list[str] | None = None) -> pd.DataFrame:
    df_active = generate_active_production(rules)
    if df_active.empty:
        raise ValueError("Aucune donnée générée à partir des règles (df_active vide).")

    df_full = complete_years(df_active, START_YEAR, END_YEAR)
    df_final = compute_hydrogen_demand(df_full)
    df_final = sort_for_display(df_final, techno_order=techno_order)

    df_final = df_final[STEEL_COLS]
    df_final.to_csv(STEEL_CSV, index=False)
    return df_final

In [121]:
# CHUNK 9 — Exécution + checks simples
#---------------------------------------------
steel_df = build_and_write_steel(ABS_RULES, techno_order=TECHNO_ORDER)

print("steel.csv généré :", len(steel_df), "lignes")
display(steel_df.head(30))

# Check: toutes les années 2020..2050 sont présentes pour chaque (area, techno)
years_expected = set(range(START_YEAR, END_YEAR + 1))
missing = (steel_df.groupby(["area", "techno"])["year"]
           .apply(lambda s: years_expected - set(s.tolist()))
           .reset_index(name="missing_years"))
missing = missing[missing["missing_years"].apply(len) > 0]
print("Groupes (area,techno) avec années manquantes:", len(missing))
if len(missing) > 0:
    display(missing.head(20))

steel.csv généré : 589 lignes


,area,techno,year,production_yearly,hydrogen_demand,source
0,BE,BF-BOF,2020,5250000.0,0.0,Steel BEL - Rapports 2024
1,BE,BF-BOF,2021,5250000.0,0.0,Steel BEL - Rapports 2024
2,BE,BF-BOF,2022,5250000.0,0.0,Steel BEL - Rapports 2024
3,BE,BF-BOF,2023,5250000.0,0.0,Steel BEL - Rapports 2024
4,BE,BF-BOF,2024,5250000.0,0.0,Steel BEL - Rapports 2024
5,BE,BF-BOF,2025,5250000.0,0.0,Steel BEL - Rapports 2024
6,BE,BF-BOF,2026,5250000.0,0.0,Steel BEL - Rapports 2024
7,BE,BF-BOF,2027,5250000.0,0.0,Steel BEL - Rapports 2024
8,BE,BF-BOF,2028,5250000.0,0.0,Steel BEL - Rapports 2024
9,BE,BF-BOF,2029,5250000.0,0.0,Steel BEL - Rapports 2024


Groupes (area,techno) avec années manquantes: 0
